# Gemma 4 31B Instruct — Final TPU One-Shot

Run All only after G9 is CLOSED/PASS and the deliberate Kaggle restart. This notebook is configuration, exact-source checkout, and one tracked-orchestrator call; it contains no model implementation.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/dangkhoa2016/KerasHub-Gemma4-31B-IT-Kaggle-TPU-v5e8-Text-Vision.git'
FINAL_TPU_EXECUTION_SHA = os.environ.get('FINAL_TPU_EXECUTION_SHA', '').strip()
G9_EVIDENCE_DIR = Path(os.environ.get('G9_EVIDENCE_DIR', '/kaggle/working/artifacts/g9'))
G10_EVIDENCE_DIR = Path(os.environ.get('G10_EVIDENCE_DIR', '/kaggle/working/artifacts/g10'))
MODEL_PRESET = 'gemma4_instruct_31b'
KERAS_BACKEND = 'jax'
MODEL_DTYPE = 'bfloat16'
EXPECTED_TPU_DEVICES = 8
MESH_SHAPE = [1, 8]
MESH_AXES = ['batch', 'model']
assert len(FINAL_TPU_EXECUTION_SHA) == 40, 'Set FINAL_TPU_EXECUTION_SHA to the CI-verified execution SHA'
checkpoint = G9_EVIDENCE_DIR / '00-context.txt'
assert checkpoint.is_file(), f'Missing pre-restart G9 checkpoint: {checkpoint}'
print('REPO_URL=' + REPO_URL)
print('FINAL_TPU_EXECUTION_SHA=' + FINAL_TPU_EXECUTION_SHA)
print('MODEL_PRESET=' + MODEL_PRESET)
print('G9_RESTART_CHECKPOINT=' + str(checkpoint))

In [ ]:
# Restore the exact clean execution source into a fresh path after restart.
import subprocess

source = Path('/kaggle/working/gemma4-final-run-all-source')
if not (source / '.git').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(source)], check=True)
subprocess.run(['git', '-C', str(source), 'fetch', '--prune', 'origin'], check=True)
subprocess.run(['git', '-C', str(source), 'checkout', '--detach', FINAL_TPU_EXECUTION_SHA], check=True)
head = subprocess.check_output(['git', '-C', str(source), 'rev-parse', 'HEAD'], text=True).strip()
assert head == FINAL_TPU_EXECUTION_SHA, (head, FINAL_TPU_EXECUTION_SHA)
assert not subprocess.check_output(['git', '-C', str(source), 'status', '--porcelain'], text=True).strip()
print('GIT_SHA_EXACT=true')
print('SOURCE_ROOT=' + str(source))

In [ ]:
# The tracked orchestrator owns all TPU/model/REST/evidence behavior.
import subprocess

run_env = os.environ.copy()
run_env.update({
    'FINAL_TPU_EXECUTION_SHA': FINAL_TPU_EXECUTION_SHA,
    'G10_SESSION_FRESH': 'true',
    'PYTHONPATH': str(source / 'src') + os.pathsep + run_env.get('PYTHONPATH', ''),
})
subprocess.run([
    'python3', str(source / 'scripts' / 'final_tpu_one_shot.py'),
    '--mode', 'g10',
    '--expected-sha', FINAL_TPU_EXECUTION_SHA,
    '--repo', str(source),
    '--evidence-dir', str(G10_EVIDENCE_DIR),
    '--restart-checkpoint', str(checkpoint),
], env=run_env, check=True)
print((G10_EVIDENCE_DIR / '12-final-adjudication.txt').read_text())

The orchestrator must print `G10_STATUS=CLOSED/PASS`; otherwise the handoff is not valid. Evidence packaging excludes weights, caches, credentials, PID files, runtime databases, and private logs.